## 01 — Preprocessing (interactive, spec-driven)

This notebook implements the **offline preprocessing pipeline** from `technical-specification.pdf` (Raw MIDI → ready-to-train tensors), step by step.

### Spec checklist (we will implement in order)

- **Inputs**: `data/maestro-v3.0.0-midi.zip` + `data/surname_checked_midis_v1.2.zip`
- **Filter to 4/4-only**: allow `2/2`, `2/4`, `4/4`
- **Tokenizer**: `miditok.REMI` with:
  - `pitch_range=(21, 108)`
  - `beat_res={(0,4):4, (4,8):2}`
  - `num_velocities=32`
  - `use_time_signature=False`, `use_sustain_pedals=True`, `use_tempos=False`
  - vocab size **163**; special tokens include `<PAD>`, `<BOS>`, `<Bar>`
- **Strip leading silence** (piece-level)
- **12-key transposition augmentation** (piece-level; discard if any note goes outside `[21,108]`)
- **Chunking**: fixed **24-bar** chunks
- **Block size**: `1024` tokens
  - drop `>1024`
  - pad `<1024` with `<PAD>`
  - enforce chunks end on `<Bar>` boundary
- **Per-chunk tensors**:
  - `X`: `[1024]` token ids
  - `Y`: `[1024]` shifted next-token target
  - `bar_indices`: `[1024]` values `0..23`
  - `attributes`: `[24,4]` bins `0..7`
- **Attributes** (per bar): Polyphony Rate, Rhythmic Intensity, Velocity Dynamics, Note Density
- **Quantile binning**: fit on **TRAIN** set; 8 bins (`0..7`); reuse bins across transpositions
- **Split**: train/val/test = **80/10/10** (piece-level)

We’ll keep each stage small and verifiable, and persist intermediate artifacts under `artifacts/`.


In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

REPO_ROOT = Path("..").resolve()  # notebook runs from notebooks/
DATA_DIR = REPO_ROOT / "data"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"

INDEX_DIR = ARTIFACTS_DIR / "index"
SPLITS_DIR = ARTIFACTS_DIR / "splits"
ATTR_DIR = ARTIFACTS_DIR / "attributes"
PREPROC_DIR = ARTIFACTS_DIR / "preprocessed"

MAESTRO_ZIP = DATA_DIR / "maestro-v3.0.0-midi.zip"
SURNAME_ZIP = DATA_DIR / "surname_checked_midis_v1.2.zip"

ALLOWED_TIME_SIGNATURES = {(2, 2), (2, 4), (4, 4)}
BARS_PER_SAMPLE = 24
BLOCK_SIZE = 1024
PITCH_RANGE = (21, 108)
NUM_VELOCITIES = 32

# For deterministic splits / sampling in notebook experiments
RNG_SEED = 440

print("repo:", REPO_ROOT)
print("data:", DATA_DIR)
print("artifacts:", ARTIFACTS_DIR)
print("maestro zip exists:", MAESTRO_ZIP.exists())
print("surname zip exists:", SURNAME_ZIP.exists())


repo: /Users/kshoker/Desktop/CPSC 440/project
data: /Users/kshoker/Desktop/CPSC 440/project/data
artifacts: /Users/kshoker/Desktop/CPSC 440/project/artifacts
maestro zip exists: True
surname zip exists: True


## Step 2 — Load existing index/filter artifacts (reuse what’s already done)

Your repo already contains early-stage artifacts under `artifacts/index/`:

- `midi_index.jsonl` (per MIDI metadata; includes time signatures + leading silence ticks)
- `midi_filtered.jsonl` (filtered list)
- `midi_index_summary.json`, `midi_filter_summary.json` (counts)

We’ll load these and sanity-check them first. If anything is missing, we’ll regenerate later.


In [2]:
import math
from typing import Iterable


def read_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


midi_index_path = INDEX_DIR / "midi_index.jsonl"
filtered_path = INDEX_DIR / "midi_filtered.jsonl"
index_summary_path = INDEX_DIR / "midi_index_summary.json"
filter_summary_path = INDEX_DIR / "midi_filter_summary.json"

assert midi_index_path.exists(), f"Missing {midi_index_path}"
assert filtered_path.exists(), f"Missing {filtered_path}"
assert index_summary_path.exists(), f"Missing {index_summary_path}"
assert filter_summary_path.exists(), f"Missing {filter_summary_path}"

midi_index = read_jsonl(midi_index_path)
filtered = read_jsonl(filtered_path)

index_summary = json.loads(index_summary_path.read_text(encoding="utf-8"))
filter_summary = json.loads(filter_summary_path.read_text(encoding="utf-8"))

print("index rows:", len(midi_index))
print("filtered rows:", len(filtered))
print("index_summary total_midis:", index_summary.get("total_midis"))
print("filter_summary kept:", filter_summary.get("kept"), "/", filter_summary.get("input_total"))
print("kept_per_zip:", filter_summary.get("kept_per_zip"))


index rows: 8512
filtered rows: 8512
index_summary total_midis: 8512
filter_summary kept: 8512 / 8512
kept_per_zip: {'maestro-v3.0.0-midi.zip': 1276, 'surname_checked_midis_v1.2.zip': 7236}


In [3]:
# Sanity checks: time signature allowlist + member keys

allowed_ts = {tuple(x.split("/")) for x in filter_summary["allowed_time_signatures"]}
allowed_ts = {(int(a), int(b)) for (a, b) in allowed_ts}
assert allowed_ts == ALLOWED_TIME_SIGNATURES, (allowed_ts, ALLOWED_TIME_SIGNATURES)

filtered_keys = {(r["zip"], r["member"]) for r in filtered}
index_keys = {(r["zip"], r["member"]) for r in midi_index}

missing_in_index = filtered_keys - index_keys
missing_in_filtered = index_keys - filtered_keys

print("filtered not in index:", len(missing_in_index))
print("index not in filtered:", len(missing_in_filtered))
if missing_in_index:
    print("example missing_in_index:", next(iter(missing_in_index)))
if missing_in_filtered:
    print("example missing_in_filtered:", next(iter(missing_in_filtered)))

assert len(missing_in_index) == 0, "Filtered list has items not present in midi_index"
# It’s okay if index has more than filtered (depending on how you generated them), but
# based on summary we expect they match.


filtered not in index: 0
index not in filtered: 0


## Step 3 — Deterministic train/val/test split (piece-level)

Spec requires **80/10/10**. We split at the *piece* level (unique `(zip, member)`), then all derived transpositions/chunks inherit the same split.

We’ll persist split manifests under `artifacts/splits/`.


In [4]:
import random


def stable_key(zip_name: str, member: str) -> str:
    return f"{zip_name}::{member}"


all_pieces = sorted(filtered_keys, key=lambda z: stable_key(z[0], z[1]))

rng = random.Random(RNG_SEED)
rng.shuffle(all_pieces)

n = len(all_pieces)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
# remainder goes to test

train_pieces = all_pieces[:n_train]
val_pieces = all_pieces[n_train : n_train + n_val]
test_pieces = all_pieces[n_train + n_val :]

print("total:", n)
print("train:", len(train_pieces))
print("val:", len(val_pieces))
print("test:", len(test_pieces))
print("train/val/test:", len(train_pieces) / n, len(val_pieces) / n, len(test_pieces) / n)

# Persist manifests (we’ll actually run this cell when ready)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

for split_name, items in [("train", train_pieces), ("val", val_pieces), ("test", test_pieces)]:
    out_path = SPLITS_DIR / f"{split_name}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for zip_name, member in items:
            f.write(json.dumps({"zip": zip_name, "member": member}, ensure_ascii=False) + "\n")

print("wrote:", SPLITS_DIR)


total: 8512
train: 6809
val: 851
test: 852
train/val/test: 0.7999295112781954 0.0999765037593985 0.10009398496240601
wrote: /Users/kshoker/Desktop/CPSC 440/project/artifacts/splits


## Step 4 — Tokenizer (miditok REMI)

We will instantiate the tokenizer exactly as specified:

- `pitch_range=(21,108)`
- `beat_res={(0,4):4, (4,8):2}`
- `num_velocities=32`
- `use_time_signature=False`
- `use_sustain_pedals=True`
- `use_tempos=False`

Next, we’ll verify we can produce token ids containing `<Bar>` boundaries.


In [8]:
# Install/import dependencies (runs inside this notebook kernel).
# We keep it lightweight and only install if missing.

import sys
import subprocess
import inspect
from collections import Counter


def _ensure_import(pkg: str, import_name: str | None = None) -> None:
    name = import_name or pkg
    try:
        __import__(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])


_ensure_import("miditok")
_ensure_import("miditoolkit")
_ensure_import("numpy")

import numpy as np
import miditok
from miditok import REMI, TokenizerConfig

print("miditok:", getattr(miditok, "__version__", "unknown"))
print("TokenizerConfig signature:")
print(inspect.signature(TokenizerConfig))


def _make_tokenizer_config() -> TokenizerConfig:
    """Create a TokenizerConfig matching the spec 163-token grammar.

    Goal vocab components:
    - Bar: 1
    - Position: 16
    - Pitch: 88 (21..108)
    - Velocity: 32
    - Duration: 24 (from beat_res)
    - PAD, BOS: 1 each
    Total = 163

    So we must ensure ALL optional token types are disabled.
    """

    base_kwargs = dict(
        pitch_range=PITCH_RANGE,
        beat_res={(0, 4): 4, (4, 8): 2},
        num_velocities=NUM_VELOCITIES,
        use_time_signatures=False,
        use_tempos=False,
        # Spec: sustain is used but folded into Duration tokens.
        use_sustain_pedals=False,
    )

    # Build kwargs in a version-tolerant way:
    sig = inspect.signature(TokenizerConfig)
    params = set(sig.parameters.keys())

    cfg_kwargs = dict(base_kwargs)

    # Spec only needs PAD + BOS (no EOS/MASK in the 163-token count).
    if "special_tokens" in params:
        cfg_kwargs["special_tokens"] = ["PAD", "BOS"]

    # Critical to match spec vocab size:
    # - disable drum pitch tokens (they add PitchDrum_*)
    # - avoid explicit Pedal tokens (fold sustain into Duration)
    if "use_pitchdrum_tokens" in params:
        cfg_kwargs["use_pitchdrum_tokens"] = False

    # In this miditok version, folding sustain into duration is controlled by `sustain_pedal_duration`.
    if "sustain_pedal_duration" in params:
        cfg_kwargs["sustain_pedal_duration"] = True

    # Also keep other optional families off.
    for k, v in {
        "use_chords": False,
        "use_rests": False,
        "use_programs": False,
        "use_pitch_bends": False,
        "use_pitch_intervals": False,
    }.items():
        if k in params:
            cfg_kwargs[k] = v

    return TokenizerConfig(**cfg_kwargs)


# Instantiate

tokenizer_config = _make_tokenizer_config()
tokenizer = REMI(tokenizer_config)


def vocab_type_counts(vocab: dict[str, int]) -> Counter:
    c: Counter = Counter()
    for s in vocab.keys():
        tok_type = s.split("_")[0]
        c[tok_type] += 1
    return c


counts = vocab_type_counts(tokenizer.vocab)
print("vocab_size:", len(tokenizer.vocab))
print("token types:")
for k, v in counts.most_common():
    print(f"  {k}: {v}")


def _find_token_id_by_predicate(pred) -> int:
    vocab: dict[str, int] = tokenizer.vocab  # token_str -> id
    matches = [(s, i) for (s, i) in vocab.items() if pred(s)]
    if len(matches) != 1:
        preview = sorted(matches)[:10]
        raise RuntimeError(f"Expected 1 match, got {len(matches)}. Preview: {preview}")
    return matches[0][1]


def get_special_token_ids() -> dict[str, int]:
    vocab: dict[str, int] = tokenizer.vocab

    pad_id = vocab.get("PAD_None")
    if pad_id is None:
        pad_id = _find_token_id_by_predicate(lambda s: s.startswith("PAD"))

    bos_id = vocab.get("BOS_None")
    if bos_id is None:
        bos_id = _find_token_id_by_predicate(lambda s: s.startswith("BOS"))

    bar_id = vocab.get("Bar_None")
    if bar_id is None:
        bar_id = _find_token_id_by_predicate(lambda s: s.split("_")[0] == "Bar")

    return {"pad": int(pad_id), "bos": int(bos_id), "bar": int(bar_id)}


special_ids = get_special_token_ids()
print("special_ids:", special_ids)

PAD_ID = special_ids["pad"]
BOS_ID = special_ids["bos"]
BAR_ID = special_ids["bar"]

# Hard requirement from spec
if len(tokenizer.vocab) != 163:
    raise RuntimeError(
        "Tokenizer vocab does not match spec (expected 163). "
        f"Got {len(tokenizer.vocab)}. Token-type counts: {dict(counts)}"
    )


miditok: unknown
TokenizerConfig signature:
(pitch_range: 'tuple[int, int]' = (21, 109), beat_res: 'dict[tuple[int, int], int]' = {(0, 4): 8, (4, 12): 4}, num_velocities: 'int' = 32, special_tokens: 'Sequence[str]' = ['PAD', 'BOS', 'EOS', 'MASK'], encode_ids_split: "Literal['bar', 'beat', 'no']" = 'bar', use_velocities: 'bool' = True, use_note_duration_programs: 'Sequence[int]' = [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127], use_chords: 'bool' = False, use_rests: 'bool' = False, use_tempos: 'bool' = False

## Step 5 — Per-piece preprocessing loop (load → strip silence → transpose → tokenize)

For each `(zip, member)` piece:

- load MIDI from the zip
- strip leading silence (trim until first note-on)
- generate up to 12 transpositions (discard those that violate pitch range)
- tokenize each transposed MIDI into REMI ids

We’ll implement this carefully and verify on a small sample before scaling.


In [15]:
from typing import Optional
import tempfile
import zipfile

from miditoolkit import MidiFile


def load_midi_from_zip(zip_path: Path, member: str) -> bytes:
    """Return raw MIDI bytes for a given zip member."""
    with zipfile.ZipFile(zip_path, "r") as z:
        return z.read(member)


def _midibytes_to_midifile(midi_bytes: bytes) -> MidiFile:
    """Parse MIDI bytes into a miditoolkit MidiFile."""
    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp:
        tmp.write(midi_bytes)
        tmp.flush()
        return MidiFile(tmp.name)


def _midifile_to_midibytes(midi: MidiFile) -> bytes:
    """Serialize a miditoolkit MidiFile to MIDI bytes."""
    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp:
        midi.dump(tmp.name)
        tmp.flush()
        tmp.seek(0)
        return tmp.read()


def strip_leading_silence(midi_bytes: bytes) -> bytes:
    """Trim initial silence until first note-on; return new MIDI bytes."""
    midi = _midibytes_to_midifile(midi_bytes)

    min_start: Optional[int] = None
    for inst in midi.instruments:
        for n in inst.notes:
            if min_start is None or n.start < min_start:
                min_start = n.start

    if min_start is None or min_start <= 0:
        return midi_bytes

    shift = min_start

    for inst in midi.instruments:
        for n in inst.notes:
            n.start = max(0, n.start - shift)
            n.end = max(0, n.end - shift)
        for cc in getattr(inst, "control_changes", []) or []:
            cc.time = max(0, cc.time - shift)
        for pb in getattr(inst, "pitch_bends", []) or []:
            pb.time = max(0, pb.time - shift)

    for tempo in getattr(midi, "tempo_changes", []) or []:
        tempo.time = max(0, tempo.time - shift)
    for ts in getattr(midi, "time_signature_changes", []) or []:
        ts.time = max(0, ts.time - shift)
    for ks in getattr(midi, "key_signature_changes", []) or []:
        ks.time = max(0, ks.time - shift)

    return _midifile_to_midibytes(midi)


def transpose_midi(midi_bytes: bytes, semitones: int, pitch_range: tuple[int, int]) -> Optional[bytes]:
    """Transpose by semitones; return None if any note goes out of range."""
    if semitones == 0:
        return midi_bytes

    midi = _midibytes_to_midifile(midi_bytes)
    lo, hi = pitch_range

    for inst in midi.instruments:
        if getattr(inst, "is_drum", False):
            continue
        for n in inst.notes:
            p = n.pitch + semitones
            if p < lo or p > hi:
                return None

    for inst in midi.instruments:
        if getattr(inst, "is_drum", False):
            continue
        for n in inst.notes:
            n.pitch += semitones

    return _midifile_to_midibytes(midi)


def tokenize_midi_bytes(midi_bytes: bytes) -> list[int]:
    """Tokenize MIDI into REMI ids with the configured tokenizer."""
    # In this miditok build, `encode` expects a symusic.Score or a MIDI file path.
    # The most robust approach is to write bytes to a temp .mid and pass the path.
    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp:
        tmp.write(midi_bytes)
        tmp.flush()

        if hasattr(tokenizer, "encode"):
            seq = tokenizer.encode(tmp.name)
        else:
            # Very old fallback
            seq = tokenizer.midi_to_tokens(tmp.name)

    if isinstance(seq, list):
        if len(seq) != 1:
            raise RuntimeError(f"Expected 1 token stream, got {len(seq)}")
        seq = seq[0]

    ids = getattr(seq, "ids", None)
    if ids is None:
        raise RuntimeError("miditok returned an object without `.ids`")

    return [int(x) for x in ids]


## Step 6 — 24-bar chunking + block size enforcement

Given a token stream for one (possibly transposed) piece:

- split into **24-bar** chunks using `<Bar>` boundaries
- drop chunks that exceed `1024` tokens
- ensure chunks end on `<Bar>` boundary
- pad shorter chunks to exactly `1024` using `<PAD>`


In [21]:
def split_into_24_bar_chunks(tokens: list[int], bar_token_id: int, bars_per_sample: int = 24) -> list[list[int]]:
    """Split a token stream into contiguous fixed-length bar chunks.

    We treat each `bar_token_id` occurrence as the *start* of a bar.
    A 24-bar chunk is the token span from the start of bar i through the end of bar i+23
    (i.e., up to but not including the next bar-start after the 24th bar).

    This ensures chunks end on a bar boundary (no partial bar at the end).
    """
    if bars_per_sample <= 0:
        raise ValueError("bars_per_sample must be > 0")

    bar_starts = [i for i, t in enumerate(tokens) if t == bar_token_id]
    if not bar_starts:
        return []

    chunks: list[list[int]] = []

    # Need at least bars_per_sample bar starts to form one chunk.
    # We create **non-overlapping** chunks: bars [0..23], then [24..47], etc.
    # For chunk starting at bar index k, we slice from bar_starts[k] to bar_starts[k+bars_per_sample] (exclusive)
    # so the chunk contains exactly `bars_per_sample` bars.
    for k in range(0, len(bar_starts) - bars_per_sample + 1, bars_per_sample):
        start = bar_starts[k]
        end = bar_starts[k + bars_per_sample] if (k + bars_per_sample) < len(bar_starts) else len(tokens)

        chunk = tokens[start:end]
        if not chunk:
            continue
        if chunk[0] != bar_token_id:
            continue

        # Sanity: should contain exactly bars_per_sample Bar tokens unless it reaches end-of-song.
        # If end-of-song, we can accept fewer full bars only if it still has exactly bars_per_sample
        # bar starts (which it does by construction). So count should still match.
        if sum(1 for t in chunk if t == bar_token_id) != bars_per_sample:
            continue

        chunks.append(chunk)

    return chunks


def enforce_block_size(
    chunk_tokens: list[int],
    pad_token_id: int,
    block_size: int = 1024,
) -> list[int] | None:
    """Pad to block_size or return None to indicate drop.

    - Drops chunks longer than block_size.
    - Pads shorter chunks with PAD to exactly block_size.
    """
    if len(chunk_tokens) > block_size:
        return None

    if len(chunk_tokens) == block_size:
        return chunk_tokens

    return chunk_tokens + [pad_token_id] * (block_size - len(chunk_tokens))


In [22]:
chunks = split_into_24_bar_chunks(ids, BAR_ID, bars_per_sample=24)
print("num 24-bar chunks:", len(chunks))
print("first chunk bars:", sum(1 for t in chunks[0] if t == BAR_ID) if chunks else None)
print("first chunk len (raw):", len(chunks[0]) if chunks else None)

x0 = enforce_block_size(chunks[0], PAD_ID, block_size=1024) if chunks else None
print("first chunk kept?:", x0 is not None)
print("first chunk len (1024):", len(x0) if x0 else None)

num 24-bar chunks: 2
first chunk bars: 24
first chunk len (raw): 1537
first chunk kept?: False
first chunk len (1024): None


## Step 7 — Compute `bar_indices` (length 1024)

We create an integer array `bar_indices[t] ∈ {0..23}` mapping each token position to its corresponding bar index. This is used at training time for GPU-fast broadcasting of bar-level condition vectors.


In [39]:
def compute_bar_indices(x_tokens: list[int], bar_token_id: int, block_size: int = 1024) -> list[int]:
    """Compute bar index per token position (0..23), length == block_size.

    Convention:
    - Each `bar_token_id` marks the **start** of a bar.
    - A valid 24-bar chunk should begin with `bar_token_id` (bar 0) and contain 24 bar tokens total.
    - Padding tokens (PAD_ID) keep the last valid bar index.
    """
    if x_tokens is None:
        raise ValueError(
            "x_tokens is None (no kept chunk found). "
            "Pick a different MIDI or search for a chunk with len<=1024 before calling compute_bar_indices()."
        )

    if len(x_tokens) != block_size:
        raise ValueError(f"Expected x_tokens length {block_size}, got {len(x_tokens)}")

    bar_indices: list[int] = []
    current_bar = -1

    for tok in x_tokens:
        if tok == bar_token_id:
            current_bar += 1
        if current_bar < 0:
            current_bar = 0
        if current_bar > 23:
            raise ValueError("Found more than 24 bars inside a 1024-token chunk")
        bar_indices.append(current_bar)

    if current_bar != 23:
        raise ValueError(f"Expected exactly 24 bars (ending at 23), got ending bar {current_bar}")

    return bar_indices


In [45]:
# Step 7 smoke test (deterministic + informative)
# Finds a kept 24-bar chunk (<=1024 tokens), then validates bar_indices.

import random
from collections import Counter


def pick_kept_example(
    *,
    max_pieces: int = 200,
    seed: int = 0,
    zip_preference: list[str] | None = None,
) -> tuple[list[int], dict]:
    rng = random.Random(seed)

    keys = list(filtered_keys)
    if zip_preference:
        # stable ordering: prefer zips in order, then everything else
        pref = {z: i for i, z in enumerate(zip_preference)}
        keys.sort(key=lambda k: (pref.get(k[0], 999), k[0], k[1]))

    rng.shuffle(keys)

    scanned_pieces = 0
    scanned_chunks = 0
    kept_chunks = 0
    min_raw = None
    max_raw = None

    for zip_name, member in keys:
        scanned_pieces += 1
        if scanned_pieces > max_pieces:
            break

        b = load_midi_from_zip(DATA_DIR / zip_name, member)
        b2 = strip_leading_silence(b)
        ids_local = tokenize_midi_bytes(b2)

        chunks_local = split_into_24_bar_chunks(ids_local, BAR_ID, bars_per_sample=24)
        for ch in chunks_local:
            scanned_chunks += 1
            raw_len = len(ch)
            min_raw = raw_len if min_raw is None else min(min_raw, raw_len)
            max_raw = raw_len if max_raw is None else max(max_raw, raw_len)

            xb = enforce_block_size(ch, PAD_ID, block_size=1024)
            if xb is not None:
                kept_chunks += 1
                meta = {
                    "zip": zip_name,
                    "member": member,
                    "ids_len": len(ids_local),
                    "num_24bar_chunks_in_piece": len(chunks_local),
                    "kept_chunk_raw_len": raw_len,
                    "scanned_pieces": scanned_pieces,
                    "scanned_chunks": scanned_chunks,
                    "kept_chunks": kept_chunks,
                    "min_raw_seen": min_raw,
                    "max_raw_seen": max_raw,
                }
                return xb, meta

    raise RuntimeError(
        "No kept chunk found. "
        f"Scanned pieces={scanned_pieces}, chunks={scanned_chunks}, kept={kept_chunks}, "
        f"min_raw={min_raw}, max_raw={max_raw}."
    )


x, meta = pick_kept_example(
    max_pieces=400,
    seed=0,
    zip_preference=["maestro-v3.0.0-midi.zip", "surname_checked_midis_v1.2.zip"],
)

bar_idx = compute_bar_indices(x, BAR_ID, block_size=1024)

print("meta:", meta)
print("x_len:", len(x))
print("bar_idx min/max:", min(bar_idx), max(bar_idx))

# Bar token count should be 24 in the kept chunk
bar_count = sum(1 for t in x if t == BAR_ID)
print("Bar tokens in x:", bar_count)

# Each bar index should appear at least once (unless padding eats the tail)
counts = Counter(bar_idx)
print("bar_idx counts (0..23):", [counts.get(i, 0) for i in range(24)])


meta: {'zip': 'surname_checked_midis_v1.2.zip', 'member': 'surname_checked_midis/Liszt, Franz, Impromptu, S.191, nLazlKjDukA.mid', 'ids_len': 5855, 'num_24bar_chunks_in_piece': 5, 'kept_chunk_raw_len': 905, 'scanned_pieces': 2, 'scanned_chunks': 8, 'kept_chunks': 1, 'min_raw_seen': 905, 'max_raw_seen': 2577}
x_len: 1024
bar_idx min/max: 0 23
Bar tokens in x: 24
bar_idx counts (0..23): [38, 35, 32, 48, 41, 36, 57, 44, 37, 32, 55, 45, 33, 21, 12, 63, 92, 17, 9, 33, 23, 37, 32, 152]


In [46]:
bar_idx = compute_bar_indices(x, BAR_ID, block_size=1024)
min(bar_idx), max(bar_idx), bar_idx[:20], bar_idx[-20:]

(0,
 23,
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23,
  23])

In [38]:
import random
from statistics import mean

def scan_chunk_lengths(n_pieces=50, seed=0, restrict_zip=None):
    rng = random.Random(seed)
    keys = [k for k in filtered_keys if (restrict_zip is None or k[0] == restrict_zip)]

    per_chunk_lens = []
    per_piece = []
    for _ in range(n_pieces):
        zip_name, member = rng.choice(keys)
        b = load_midi_from_zip(DATA_DIR / zip_name, member)
        b2 = strip_leading_silence(b)
        ids_local = tokenize_midi_bytes(b2)

        chunks_local = split_into_24_bar_chunks(ids_local, BAR_ID, bars_per_sample=24)
        lens = [len(ch) for ch in chunks_local]
        per_piece.append((zip_name, member, len(ids_local), len(chunks_local), min(lens) if lens else None))
        per_chunk_lens.extend(lens)

    kept = [L for L in per_chunk_lens if L <= 1024]
    print("pieces:", n_pieces)
    print("chunks_total:", len(per_chunk_lens))
    print("chunks_kept(<=1024):", len(kept))
    print("min_chunk_len:", min(per_chunk_lens) if per_chunk_lens else None)
    print("mean_chunk_len:", mean(per_chunk_lens) if per_chunk_lens else None)
    print("max_chunk_len:", max(per_chunk_lens) if per_chunk_lens else None)
    print("example pieces (zip, member, ids_len, num_chunks, min_chunk_len):")
    for row in per_piece[:5]:
        print("  ", row)

scan_chunk_lengths(n_pieces=50, seed=0, restrict_zip="maestro-v3.0.0-midi.zip")

pieces: 50
chunks_total: 456
chunks_kept(<=1024): 99
min_chunk_len: 242
mean_chunk_len: 1703.8070175438597
max_chunk_len: 3905
example pieces (zip, member, ids_len, num_chunks, min_chunk_len):
   ('maestro-v3.0.0-midi.zip', 'maestro-v3.0.0/2009/MIDI-Unprocessed_16_R2_2009_01_ORIG_MID--AUDIO_16_R2_2009_16_R2_2009_03_WAV.midi', 10700, 6, 1081)
   ('maestro-v3.0.0-midi.zip', 'maestro-v3.0.0/2008/MIDI-Unprocessed_17_R1_2008_01-04_ORIG_MID--AUDIO_17_R1_2008_wav--1.midi', 7505, 6, 556)
   ('maestro-v3.0.0-midi.zip', 'maestro-v3.0.0/2009/MIDI-Unprocessed_12_R1_2009_03-05_ORIG_MID--AUDIO_12_R1_2009_12_R1_2009_03_WAV.midi', 24340, 18, 476)
   ('maestro-v3.0.0-midi.zip', 'maestro-v3.0.0/2009/MIDI-Unprocessed_21_R2_2009_01_ORIG_MID--AUDIO_21_R2_2009_21_R2_2009_03_WAV.midi', 10864, 4, 2037)
   ('maestro-v3.0.0-midi.zip', 'maestro-v3.0.0/2017/MIDI-Unprocessed_051_PIANO051_MID--AUDIO-split_07-06-17_Piano-e_3-02_wav--1.midi', 4381, 3, 1129)


## Step 8 — Compute raw per-bar attributes (shape `[24,4]`)

Attributes per bar:

- **Polyphony Rate**
- **Rhythmic Intensity**
- **Velocity Dynamics**
- **Note Density**

We’ll compute raw values bar-by-bar for each 24-bar chunk and keep them *unbinned* initially.


In [52]:
ATTRIBUTE_NAMES = [
    "polyphony_rate",
    "rhythmic_intensity",
    "velocity_dynamics",
    "note_density",
]

# id -> token string mapping for parsing
ID_TO_TOKEN = {i: s for (s, i) in tokenizer.vocab.items()}


def _parse_duration_beats(token: str) -> float:
    """Parse a miditok REMI Duration token into beats.

    miditok versions use either:
    - `Duration_<float>` (e.g. `Duration_0.5`)
    - `Duration_A.B.C` (e.g. `Duration_3.2.4`) meaning \(A + B/C\) beats.
    """
    raw = token.split("_", 1)[1]

    # Common case
    try:
        return float(raw)
    except Exception:
        pass

    # Fractional encoding case: A.B.C -> A + B/C
    parts = raw.split(".")
    if len(parts) == 3:
        try:
            a = int(parts[0])
            b = int(parts[1])
            c = int(parts[2])
            if c == 0:
                raise ValueError
            return float(a + (b / c))
        except Exception as e:
            raise ValueError(f"Cannot parse duration token: {token}") from e

    raise ValueError(f"Cannot parse duration token: {token}")


def _parse_position_index(token: str) -> int:
    # e.g. 'Position_0'..'Position_15'
    try:
        return int(token.split("_", 1)[1])
    except Exception as e:
        raise ValueError(f"Cannot parse position token: {token}") from e


def _parse_velocity_bin(token: str) -> int:
    # e.g. 'Velocity_16'
    try:
        return int(token.split("_", 1)[1])
    except Exception as e:
        raise ValueError(f"Cannot parse velocity token: {token}") from e


def compute_raw_attributes_per_bar_from_tokens(
    x_tokens_1024: list[int],
    *,
    bar_token_id: int,
    bars_per_sample: int = 24,
) -> list[list[float]]:
    """Return raw per-bar metrics from REMI token ids, shape [bars][4].

    Token-derived proxies (simple + fast):
    - polyphony_rate: mean active-note count sampled at the 16 positions in the bar
    - rhythmic_intensity: fraction of positions (0..15) that contain any note onset
    - velocity_dynamics: std-dev of velocity bins for note onsets in the bar
    - note_density: number of note onsets in the bar

    This is aligned to the *exact* 24-bar chunk we train on.
    """
    if len(x_tokens_1024) != 1024:
        raise ValueError(f"Expected 1024 tokens, got {len(x_tokens_1024)}")

    import numpy as np

    # per-bar accumulators
    onsets = [0 for _ in range(bars_per_sample)]
    pos_has_onset = [set() for _ in range(bars_per_sample)]
    velocities = [[] for _ in range(bars_per_sample)]

    # For polyphony across bar boundaries: store global (start,end) in beats across full 24 bars
    # where bar 0 spans beats [0,4), bar 1 spans [4,8), etc.
    global_intervals: list[tuple[float, float]] = []

    current_bar = -1
    current_pos = 0

    i = 0
    while i < len(x_tokens_1024):
        tid = x_tokens_1024[i]
        tok = ID_TO_TOKEN.get(tid)
        if tok is None:
            i += 1
            continue

        typ = tok.split("_", 1)[0]

        if tid == bar_token_id or typ == "Bar":
            current_bar += 1
            current_pos = 0
            i += 1
            continue

        if current_bar < 0:
            i += 1
            continue
        if current_bar >= bars_per_sample:
            break

        if typ == "Position":
            current_pos = _parse_position_index(tok)
            i += 1
            continue

        # Parse note event: Pitch -> Velocity -> Duration
        if typ == "Pitch" and i + 2 < len(x_tokens_1024):
            vtok = ID_TO_TOKEN.get(x_tokens_1024[i + 1], "")
            dtok = ID_TO_TOKEN.get(x_tokens_1024[i + 2], "")
            if vtok.startswith("Velocity_") and dtok.startswith("Duration_"):
                vel = _parse_velocity_bin(vtok)
                dur_beats = _parse_duration_beats(dtok)

                onsets[current_bar] += 1
                pos_has_onset[current_bar].add(current_pos)
                velocities[current_bar].append(vel)

                # Global beat timeline across the full 24-bar chunk
                start_beat = current_bar * 4.0 + (current_pos * 0.25)  # 16 positions per 4/4 bar
                end_beat = start_beat + dur_beats
                global_intervals.append((start_beat, end_beat))

                i += 3
                continue

        i += 1

    attrs: list[list[float]] = []
    for b in range(bars_per_sample):
        note_density = float(onsets[b])
        rhythmic_intensity = float(len(pos_has_onset[b]) / 16.0)

        vel_std = float(np.std(np.array(velocities[b]), ddof=0)) if velocities[b] else 0.0

        poly_samples = []
        for p in range(16):
            # sample at each position on the global beat timeline
            t = b * 4.0 + (p * 0.25)
            active = 0
            for s, e in global_intervals:
                if s <= t < e:
                    active += 1
            poly_samples.append(active)
        polyphony_rate = float(sum(poly_samples) / 16.0)

        attrs.append([polyphony_rate, rhythmic_intensity, vel_std, note_density])

    return attrs


def compute_raw_attributes_per_bar(midi_bytes: bytes, bars_per_sample: int = 24) -> list[list[float]]:
    """Return raw per-bar metrics, shape [bars][4].

    We’ll implement the MIDI-note-based version later if needed.
    For Step 8 now, use `compute_raw_attributes_per_bar_from_tokens(x, bar_token_id=BAR_ID)`.
    """
    raise NotImplementedError(
        "Use compute_raw_attributes_per_bar_from_tokens(x_tokens_1024, bar_token_id=BAR_ID) for Step 8."
    )


<>:17: SyntaxWarning: invalid escape sequence '\('
<>:17: SyntaxWarning: invalid escape sequence '\('
/var/folders/yj/3yq26dgs2nn76q3v6zwh32mc0000gn/T/ipykernel_54100/1864970668.py:17: SyntaxWarning: invalid escape sequence '\('
  - `Duration_A.B.C` (e.g. `Duration_3.2.4`) meaning \(A + B/C\) beats.


In [53]:
# Step 8 smoke test on the same kept chunk `x`

raw_attrs = compute_raw_attributes_per_bar_from_tokens(x, bar_token_id=BAR_ID, bars_per_sample=24)

print("shape:", (len(raw_attrs), len(raw_attrs[0]) if raw_attrs else None))
print("names:", ATTRIBUTE_NAMES)
print("bar0:", raw_attrs[0])
print("bar23:", raw_attrs[23])

# quick sanity ranges
poly = [r[0] for r in raw_attrs]
rhy = [r[1] for r in raw_attrs]
vel = [r[2] for r in raw_attrs]
den = [r[3] for r in raw_attrs]

print("polyphony_rate min/max:", min(poly), max(poly))
print("rhythmic_intensity min/max:", min(rhy), max(rhy))
print("velocity_dynamics min/max:", min(vel), max(vel))
print("note_density min/max:", min(den), max(den))


shape: (24, 4)
names: ['polyphony_rate', 'rhythmic_intensity', 'velocity_dynamics', 'note_density']
bar0: [3.0625, 0.4375, 8.039900496896712, 10.0]
bar23: [6.4375, 0.5, 2.8284271247461903, 8.0]
polyphony_rate min/max: 1.5 8.875
rhythmic_intensity min/max: 0.125 1.0
velocity_dynamics min/max: 2.0 20.997354330698162
note_density min/max: 2.0 25.0


## Step 9 — Fit global quantile binning on TRAIN set only

We aggregate raw attributes across all **training** chunks and fit per-attribute quantile thresholds to define 8 bins (0–7). We persist the thresholds so val/test (and inference) can use the exact same mapping.


In [67]:
# Step 9 implementation (TRAIN-only quantiles) + quick sanity test

import numpy as np


def fit_quantile_thresholds(
    train_raw_attrs: list[list[float]],
    num_bins: int = 8,
) -> dict[str, list[float]]:
    """Return per-attribute quantile thresholds (fit on TRAIN only)."""
    if num_bins != 8:
        raise ValueError("Spec requires 8 bins (0..7)")

    arr = np.asarray(train_raw_attrs, dtype=float)
    if arr.ndim != 2 or arr.shape[1] != 4:
        raise ValueError(f"Expected shape [N,4], got {arr.shape}")
    if arr.shape[0] == 0:
        raise ValueError("No training rows provided for quantile fitting")

    quantiles = [i / num_bins for i in range(1, num_bins)]  # 1/8..7/8

    thresholds: dict[str, list[float]] = {}
    for j, name in enumerate(ATTRIBUTE_NAMES):
        th = np.quantile(arr[:, j], quantiles, method="linear")
        thresholds[name] = [float(x) for x in th.tolist()]
    return thresholds


def bin_attributes(
    raw_attrs_24x4: list[list[float]],
    thresholds: dict[str, list[float]],
) -> list[list[int]]:
    """Map raw attrs to integer bins 0..7; return shape [24][4]."""
    arr = np.asarray(raw_attrs_24x4, dtype=float)
    if arr.ndim != 2 or arr.shape != (24, 4):
        raise ValueError(f"Expected shape [24,4], got {arr.shape}")

    out = np.zeros_like(arr, dtype=int)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        th = np.asarray(thresholds[name], dtype=float)
        if th.shape != (7,):
            raise ValueError(f"Expected 7 thresholds for {name}, got shape {th.shape}")
        out[:, j] = np.digitize(arr[:, j], th, right=False)

    return out.astype(int).tolist()


# --- quick test on a small TRAIN sample ---

def load_split(path: Path) -> list[tuple[str, str]]:
    rows = read_jsonl(path)
    return [(r["zip"], r["member"]) for r in rows]


train_keys = load_split(SPLITS_DIR / "train.jsonl")

rng = random.Random(0)
# sample a few pieces so this runs quickly
sample_keys = [train_keys[i] for i in rng.sample(range(len(train_keys)), 30)]

train_bar_rows: list[list[float]] = []
for zip_name, member in sample_keys:
    b = load_midi_from_zip(DATA_DIR / zip_name, member)
    b2 = strip_leading_silence(b)
    ids_local = tokenize_midi_bytes(b2)

    chunks_local = split_into_24_bar_chunks(ids_local, BAR_ID, bars_per_sample=24)
    for ch in chunks_local:
        xb = enforce_block_size(ch, PAD_ID, 1024)
        if xb is None:
            continue
        raw_24x4 = compute_raw_attributes_per_bar_from_tokens(xb, bar_token_id=BAR_ID, bars_per_sample=24)
        train_bar_rows.extend(raw_24x4)

print("train bars collected:", len(train_bar_rows))

thresholds = fit_quantile_thresholds(train_bar_rows, num_bins=8)
print("thresholds keys:", list(thresholds.keys()))
print("polyphony thresholds:", thresholds["polyphony_rate"])

# bin current example x
raw_x = compute_raw_attributes_per_bar_from_tokens(x, bar_token_id=BAR_ID, bars_per_sample=24)
binned_x = bin_attributes(raw_x, thresholds)

mins = [min(row[j] for row in binned_x) for j in range(4)]
maxs = [max(row[j] for row in binned_x) for j in range(4)]
print("binned min per attr:", mins)
print("binned max per attr:", maxs)


train bars collected: 1968
thresholds keys: ['polyphony_rate', 'rhythmic_intensity', 'velocity_dynamics', 'note_density']
polyphony thresholds: [0.0625, 1.3125, 2.25, 2.9375, 3.5, 4.4375, 5.9375]
binned min per attr: [2, 3, 2, 2]
binned max per attr: [7, 7, 7, 7]


## Step 10 — Apply binning to train/val/test; verify transposition invariance

We apply the train-fit thresholds to all splits, and check that transposed versions of the same chunk reuse the same binned labels (since the attributes are pitch-invariant).


In [68]:
# Step 10: apply binning to splits + transposition invariance spot-check

from collections import defaultdict


def summarize_split(
    split_name: str,
    *,
    thresholds: dict[str, list[float]],
    n_pieces: int = 30,
    seed: int = 0,
) -> dict:
    keys = load_split(SPLITS_DIR / f"{split_name}.jsonl")
    rng = random.Random(seed)
    sample_keys = [keys[i] for i in rng.sample(range(len(keys)), min(n_pieces, len(keys)))]

    total_chunks = 0
    kept_chunks = 0
    binned_counts = [defaultdict(int) for _ in range(4)]  # attr -> bin -> count

    for zip_name, member in sample_keys:
        b = load_midi_from_zip(DATA_DIR / zip_name, member)
        b2 = strip_leading_silence(b)
        ids_local = tokenize_midi_bytes(b2)
        chunks_local = split_into_24_bar_chunks(ids_local, BAR_ID, bars_per_sample=24)

        for ch in chunks_local:
            total_chunks += 1
            xb = enforce_block_size(ch, PAD_ID, 1024)
            if xb is None:
                continue
            kept_chunks += 1

            raw_24x4 = compute_raw_attributes_per_bar_from_tokens(xb, bar_token_id=BAR_ID, bars_per_sample=24)
            binned_24x4 = bin_attributes(raw_24x4, thresholds)

            for bar in range(24):
                for j in range(4):
                    binned_counts[j][binned_24x4[bar][j]] += 1

    # Turn into simple summaries
    per_attr = {}
    for j, name in enumerate(ATTRIBUTE_NAMES):
        bins_present = sorted(binned_counts[j].keys())
        per_attr[name] = {
            "bins_present": bins_present,
            "min_bin": min(bins_present) if bins_present else None,
            "max_bin": max(bins_present) if bins_present else None,
        }

    return {
        "split": split_name,
        "sample_pieces": len(sample_keys),
        "total_24bar_chunks": total_chunks,
        "kept_24bar_chunks": kept_chunks,
        "kept_ratio": (kept_chunks / total_chunks) if total_chunks else None,
        "per_attr": per_attr,
    }


summary_train = summarize_split("train", thresholds=thresholds, n_pieces=30, seed=0)
summary_val = summarize_split("val", thresholds=thresholds, n_pieces=30, seed=1)
summary_test = summarize_split("test", thresholds=thresholds, n_pieces=30, seed=2)

print("train summary:", summary_train)
print("val summary:", summary_val)
print("test summary:", summary_test)


# --- Transposition invariance spot-check (robust) ---
# We search for a TRAIN piece with at least one kept chunk, then compare the SAME chunk index
# across transpositions (skipping transpositions where that chunk index is dropped).

def find_invariance_anchor(
    *,
    max_pieces: int = 200,
    seed: int = 0,
) -> tuple[str, str, int, list[int]]:
    rng = random.Random(seed)
    keys = load_split(SPLITS_DIR / "train.jsonl")
    rng.shuffle(keys)

    for zip_name, member in keys[:max_pieces]:
        b = load_midi_from_zip(DATA_DIR / zip_name, member)
        b2 = strip_leading_silence(b)
        ids_base = tokenize_midi_bytes(b2)
        chunks_base = split_into_24_bar_chunks(ids_base, BAR_ID, bars_per_sample=24)

        for chunk_idx, ch in enumerate(chunks_base):
            xb = enforce_block_size(ch, PAD_ID, 1024)
            if xb is not None:
                return zip_name, member, chunk_idx, xb

    raise RuntimeError(f"No invariance anchor found after scanning {max_pieces} train pieces")


zip_name, member, chunk_idx, base_x = find_invariance_anchor(max_pieces=300, seed=0)
zip_path = DATA_DIR / zip_name

b = load_midi_from_zip(zip_path, member)
b2 = strip_leading_silence(b)

raw_base = compute_raw_attributes_per_bar_from_tokens(base_x, bar_token_id=BAR_ID, bars_per_sample=24)
binned_base = bin_attributes(raw_base, thresholds)

ok = True
checked = 0
skipped = 0

for semitones in [-5, -3, -1, 1, 3, 5]:
    bt = transpose_midi(b2, semitones=semitones, pitch_range=PITCH_RANGE)
    if bt is None:
        skipped += 1
        continue

    ids_t = tokenize_midi_bytes(bt)
    chunks_t = split_into_24_bar_chunks(ids_t, BAR_ID, bars_per_sample=24)

    if chunk_idx >= len(chunks_t):
        skipped += 1
        continue

    xt = enforce_block_size(chunks_t[chunk_idx], PAD_ID, 1024)
    if xt is None:
        skipped += 1
        continue

    raw_t = compute_raw_attributes_per_bar_from_tokens(xt, bar_token_id=BAR_ID, bars_per_sample=24)
    binned_t = bin_attributes(raw_t, thresholds)

    checked += 1
    if binned_t != binned_base:
        ok = False
        print("invariance FAILED at semitones", semitones)
        break

print("invariance anchor:", {"zip": zip_name, "member": member, "chunk_idx": chunk_idx})
print("transposition invariance (binned attrs):", ok, "checked:", checked, "skipped:", skipped)


train summary: {'split': 'train', 'sample_pieces': 30, 'total_24bar_chunks': 292, 'kept_24bar_chunks': 82, 'kept_ratio': 0.2808219178082192, 'per_attr': {'polyphony_rate': {'bins_present': [0, 1, 2, 3, 4, 5, 6, 7], 'min_bin': 0, 'max_bin': 7}, 'rhythmic_intensity': {'bins_present': [1, 2, 3, 4, 5, 6, 7], 'min_bin': 1, 'max_bin': 7}, 'velocity_dynamics': {'bins_present': [1, 2, 3, 4, 5, 6, 7], 'min_bin': 1, 'max_bin': 7}, 'note_density': {'bins_present': [1, 2, 3, 4, 5, 6, 7], 'min_bin': 1, 'max_bin': 7}}}
val summary: {'split': 'val', 'sample_pieces': 30, 'total_24bar_chunks': 293, 'kept_24bar_chunks': 69, 'kept_ratio': 0.2354948805460751, 'per_attr': {'polyphony_rate': {'bins_present': [0, 1, 2, 3, 4, 5, 6, 7], 'min_bin': 0, 'max_bin': 7}, 'rhythmic_intensity': {'bins_present': [1, 2, 3, 4, 5, 6, 7], 'min_bin': 1, 'max_bin': 7}, 'velocity_dynamics': {'bins_present': [1, 2, 3, 4, 5, 6, 7], 'min_bin': 1, 'max_bin': 7}, 'note_density': {'bins_present': [1, 2, 3, 4, 5, 6, 7], 'min_bin': 1

## Step 11 — Build final tensors (`X`, `Y`, `bar_indices`, `attributes`)

For each kept chunk:

- `X`: `[1024]` token ids (padded)
- `Y`: `[1024]` next-token targets (shifted)
- `bar_indices`: `[1024]` int bar mapping
- `attributes`: `[24,4]` int bins

We’ll standardize exact behavior for the final position in `Y` (e.g., `PAD` or `BAR`).


In [69]:
def build_xy(x_tokens_1024: list[int], pad_token_id: int) -> tuple[list[int], list[int]]:
    """Return (X, Y) for autoregressive next-token prediction.

    - X is the input token sequence (length 1024)
    - Y is X shifted left by 1
    - The final target token is PAD (common practice for fixed-block training)
    """
    if len(x_tokens_1024) != 1024:
        raise ValueError(f"Expected X length 1024, got {len(x_tokens_1024)}")

    x = list(x_tokens_1024)
    y = x[1:] + [pad_token_id]
    return x, y


In [70]:
# Step 11 smoke test: build X/Y/bar_indices/attributes for current example chunk `x`

X, Y = build_xy(x, PAD_ID)
bar_indices = compute_bar_indices(X, BAR_ID, block_size=1024)
raw_attrs = compute_raw_attributes_per_bar_from_tokens(X, bar_token_id=BAR_ID, bars_per_sample=24)
attributes = bin_attributes(raw_attrs, thresholds)

print("X shape:", (len(X),))
print("Y shape:", (len(Y),))
print("bar_indices shape:", (len(bar_indices),), "min/max:", min(bar_indices), max(bar_indices))
print("attributes shape:", (len(attributes), len(attributes[0]) if attributes else None))

# Basic sanity checks
assert len(X) == 1024 and len(Y) == 1024
assert len(bar_indices) == 1024
assert len(attributes) == 24 and all(len(r) == 4 for r in attributes)
assert all(0 <= v <= 7 for r in attributes for v in r)

print("Y[0] == X[1]:", Y[0] == X[1])
print("Y[-1] == PAD_ID:", Y[-1] == PAD_ID)


X shape: (1024,)
Y shape: (1024,)
bar_indices shape: (1024,) min/max: 0 23
attributes shape: (24, 4)
Y[0] == X[1]: True
Y[-1] == PAD_ID: True


## Step 12 — Export shards + manifest

Spec suggests saving as flat PyTorch `.pt` files ("batch_001.pt"). We will:

- write shard files under `artifacts/preprocessed/`
- write a manifest + summary statistics (kept/dropped, histograms)


In [71]:
def export_pt_shards(examples: list[dict], out_dir: Path, shard_size: int = 2048) -> None:
    """Write examples to .pt shards and save a manifest.

    Expected per-example keys:
    - X: list[int] length 1024
    - Y: list[int] length 1024
    - bar_indices: list[int] length 1024
    - attributes: list[list[int]] shape [24][4]

    Writes:
    - shard_00000.pt, shard_00001.pt, ... (each a dict of stacked tensors)
    - manifest.json with metadata
    """
    import time

    try:
        import torch
    except Exception as e:
        raise RuntimeError(
            "PyTorch is required to export .pt shards. Install with `pip install torch`."
        ) from e

    out_dir.mkdir(parents=True, exist_ok=True)

    n = len(examples)
    if n == 0:
        raise ValueError("No examples provided")

    num_shards = (n + shard_size - 1) // shard_size
    shard_files: list[str] = []

    for shard_idx in range(num_shards):
        start = shard_idx * shard_size
        end = min(n, (shard_idx + 1) * shard_size)
        batch = examples[start:end]

        Xs = [ex["X"] for ex in batch]
        Ys = [ex["Y"] for ex in batch]
        BIs = [ex["bar_indices"] for ex in batch]
        As = [ex["attributes"] for ex in batch]

        shard = {
            "X": torch.tensor(Xs, dtype=torch.long),
            "Y": torch.tensor(Ys, dtype=torch.long),
            "bar_indices": torch.tensor(BIs, dtype=torch.long),
            "attributes": torch.tensor(As, dtype=torch.long),
        }

        if shard["X"].shape[1] != 1024 or shard["Y"].shape[1] != 1024 or shard["bar_indices"].shape[1] != 1024:
            raise ValueError("Unexpected sequence length in shard")
        if tuple(shard["attributes"].shape[1:]) != (24, 4):
            raise ValueError("Unexpected attributes shape in shard")

        fname = f"shard_{shard_idx:05d}.pt"
        torch.save(shard, out_dir / fname)
        shard_files.append(fname)

    manifest = {
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "num_examples": n,
        "shard_size": shard_size,
        "num_shards": num_shards,
        "files": shard_files,
        "tensor_shapes": {
            "X": [1024],
            "Y": [1024],
            "bar_indices": [1024],
            "attributes": [24, 4],
        },
    }

    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("wrote", num_shards, "shards to", out_dir)


In [72]:
# Step 12 smoke test: export a tiny shard batch

# Build a small list of examples from the current x (and a few more from the invariance anchor if available)
examples = []

# always include current example x
X, Y = build_xy(x, PAD_ID)
examples.append({
    "X": X,
    "Y": Y,
    "bar_indices": compute_bar_indices(X, BAR_ID, 1024),
    "attributes": bin_attributes(
        compute_raw_attributes_per_bar_from_tokens(X, bar_token_id=BAR_ID, bars_per_sample=24),
        thresholds,
    ),
})

# add a few more examples by sampling random train pieces
rng = random.Random(0)
train_keys = load_split(SPLITS_DIR / "train.jsonl")
for zip_name, member in [train_keys[i] for i in rng.sample(range(len(train_keys)), 10)]:
    b = load_midi_from_zip(DATA_DIR / zip_name, member)
    b2 = strip_leading_silence(b)
    ids_local = tokenize_midi_bytes(b2)
    chunks_local = split_into_24_bar_chunks(ids_local, BAR_ID, 24)
    for ch in chunks_local:
        xb = enforce_block_size(ch, PAD_ID, 1024)
        if xb is None:
            continue
        X, Y = build_xy(xb, PAD_ID)
        examples.append({
            "X": X,
            "Y": Y,
            "bar_indices": compute_bar_indices(X, BAR_ID, 1024),
            "attributes": bin_attributes(
                compute_raw_attributes_per_bar_from_tokens(X, bar_token_id=BAR_ID, bars_per_sample=24),
                thresholds,
            ),
        })
        if len(examples) >= 32:
            break
    if len(examples) >= 32:
        break

print("examples:", len(examples))

out_dir = PREPROC_DIR / "debug_shards"
export_pt_shards(examples, out_dir=out_dir, shard_size=16)


examples: 10
wrote 1 shards to /Users/kshoker/Desktop/CPSC 440/project/artifacts/preprocessed/debug_shards


## Step 13 — Validation

We will validate the exported data by:

- reloading a few shards
- verifying shapes/dtypes and value ranges
- decoding a few token sequences back to MIDI for a quick smoke test


In [73]:
def validate_shards(preproc_dir: Path) -> None:
    """Validate exported .pt shards + manifest.

    Checks:
    - manifest exists and lists shard files
    - shard tensor shapes/dtypes
    - value ranges for bar_indices and attributes
    """
    import torch

    manifest_path = preproc_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing manifest: {manifest_path}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    files = manifest.get("files")
    if not files:
        raise ValueError("Manifest has no shard files")

    first = preproc_dir / files[0]
    if not first.exists():
        raise FileNotFoundError(f"Missing shard file listed in manifest: {first}")

    shard = torch.load(first, map_location="cpu")
    for k in ["X", "Y", "bar_indices", "attributes"]:
        if k not in shard:
            raise KeyError(f"Shard missing key {k}")

    X = shard["X"]
    Y = shard["Y"]
    BI = shard["bar_indices"]
    A = shard["attributes"]

    print("loaded shard:", first.name)
    print("X:", tuple(X.shape), X.dtype)
    print("Y:", tuple(Y.shape), Y.dtype)
    print("bar_indices:", tuple(BI.shape), BI.dtype)
    print("attributes:", tuple(A.shape), A.dtype)

    if X.ndim != 2 or X.shape[1] != 1024:
        raise ValueError("X must be [N,1024]")
    if Y.shape != X.shape:
        raise ValueError("Y must match X shape")
    if BI.shape != X.shape:
        raise ValueError("bar_indices must match X shape")
    if A.ndim != 3 or tuple(A.shape[1:]) != (24, 4):
        raise ValueError("attributes must be [N,24,4]")

    # Range checks
    bi_min = int(BI.min().item())
    bi_max = int(BI.max().item())
    if bi_min < 0 or bi_max > 23:
        raise ValueError(f"bar_indices out of range: {bi_min}..{bi_max}")

    a_min = int(A.min().item())
    a_max = int(A.max().item())
    if a_min < 0 or a_max > 7:
        raise ValueError(f"attributes out of range: {a_min}..{a_max}")

    # Shift check on first row
    if not torch.equal(Y[0, :-1], X[0, 1:]):
        raise ValueError("Y is not X shifted by 1 (row 0)")

    print("OK: shard validation passed")


In [74]:
# Step 13 smoke test: validate the debug shards we just wrote

validate_shards(PREPROC_DIR / "debug_shards")


loaded shard: shard_00000.pt
X: (10, 1024) torch.int64
Y: (10, 1024) torch.int64
bar_indices: (10, 1024) torch.int64
attributes: (10, 24, 4) torch.int64
OK: shard validation passed


## Listen to a decoded sample (end-to-end sanity check)

MIDI is not audio. We will:

1. Decode one exported `X` back into a `.mid`
2. Try to render it to `.wav` (MuseScore if available; otherwise FluidSynth if available + a SoundFont)
3. Play the `.wav` inline (if created)


In [77]:
import shutil
from pathlib import Path

import torch


def decode_example_to_midi(
    *,
    shard_dir: Path,
    example_idx: int = 0,
    out_mid: Path | None = None,
) -> Path:
    """Decode one exported X back into a MIDI file.

    miditok decode APIs vary across versions. We try a few compatible formats.
    """
    manifest = json.loads((shard_dir / "manifest.json").read_text(encoding="utf-8"))
    shard_path = shard_dir / manifest["files"][0]
    shard = torch.load(shard_path, map_location="cpu")

    X_ids: list[int] = shard["X"][example_idx].tolist()

    score = None
    decode_errors = []

    # Attempt 1: 2D ids ([I,T]) as list[list[int]]
    # Your miditok build expects i/o format (('I','T')), so we provide a single stream.
    try:
        score = tokenizer.decode([X_ids])
    except Exception as e:
        decode_errors.append(("decode(list[list[int]])", repr(e)))

    # Attempt 2: 2D ids as numpy array
    if score is None:
        try:
            import numpy as _np

            score = tokenizer.decode(_np.asarray([X_ids], dtype=int))
        except Exception as e:
            decode_errors.append(("decode(np.array([ids]))", repr(e)))

    # Attempt 3: TokSequence(ids=...)
    if score is None:
        try:
            from miditok import TokSequence

            seq = TokSequence(ids=[X_ids])
            score = tokenizer.decode(seq)
        except Exception as e:
            decode_errors.append(("decode(TokSequence(ids=[ids]))", repr(e)))

    # Attempt 4: TokSequence(tokens=...)
    if score is None:
        try:
            from miditok import TokSequence

            toks = [ID_TO_TOKEN[int(i)] for i in X_ids]
            seq = TokSequence(tokens=[toks])
            score = tokenizer.decode(seq)
        except Exception as e:
            decode_errors.append(("decode(TokSequence(tokens=[...]))", repr(e)))

    if score is None:
        print("Decode failed. Attempts:")
        for name, err in decode_errors:
            print("-", name, err)
        raise RuntimeError("Could not decode tokens back to MIDI with current miditok build")

    if out_mid is None:
        out_mid = shard_dir / f"decoded_example_{example_idx:03d}.mid"

    score.dump_midi(str(out_mid))
    print("wrote MIDI:", out_mid)
    return out_mid


decoded_mid = decode_example_to_midi(shard_dir=PREPROC_DIR / "debug_shards", example_idx=0)


wrote MIDI: /Users/kshoker/Desktop/CPSC 440/project/artifacts/preprocessed/debug_shards/decoded_example_000.mid


In [78]:
import subprocess


def render_midi_to_wav(
    mid_path: Path,
    *,
    out_wav: Path | None = None,
    soundfont_path: Path | None = None,
) -> Path:
    """Render MIDI to WAV.

    Prefers MuseScore if installed. Falls back to FluidSynth if installed and soundfont_path is provided.
    """
    if out_wav is None:
        out_wav = mid_path.with_suffix(".wav")

    # Option A: MuseScore
    musescore = shutil.which("musescore") or shutil.which("mscore") or shutil.which("MuseScore")
    if musescore:
        # MuseScore 4 uses: musescore -o out.wav in.mid
        subprocess.check_call([musescore, "-o", str(out_wav), str(mid_path)])
        print("rendered via MuseScore:", out_wav)
        return out_wav

    # Option B: FluidSynth
    fluidsynth = shutil.which("fluidsynth")
    if fluidsynth and soundfont_path:
        subprocess.check_call(
            [
                fluidsynth,
                "-ni",
                str(soundfont_path),
                str(mid_path),
                "-F",
                str(out_wav),
                "-r",
                "44100",
            ]
        )
        print("rendered via FluidSynth:", out_wav)
        return out_wav

    raise RuntimeError(
        "Could not render MIDI to WAV. Install MuseScore (preferred), or install fluidsynth and provide a .sf2 SoundFont path."
    )


# FluidSynth requires a SoundFont (.sf2). Put one at:
#   artifacts/soundfonts/GeneralUser_GS.sf2
# Then set the path below.
SOUNDFONT = ARTIFACTS_DIR / "soundfonts" / "GeneralUser_GS.sf2"

try:
    rendered_wav = render_midi_to_wav(decoded_mid, soundfont_path=SOUNDFONT)
except Exception as e:
    print("Render skipped:", e)
    rendered_wav = None


Render skipped: Could not render MIDI to WAV. Install MuseScore (preferred), or install fluidsynth and provide a .sf2 SoundFont path.


In [79]:
from IPython.display import Audio, display

if rendered_wav is not None and Path(rendered_wav).exists():
    display(Audio(filename=str(rendered_wav)))
else:
    print("No WAV rendered yet.")


No WAV rendered yet.


## What to run (the only cells that matter for full data)

1. **Fit / write quantiles** (the cell immediately above the parallel section)
2. **Parallel full preprocessing** (the 2 code cells right below this header)

Everything earlier in the notebook is just step-by-step verification utilities.


In [84]:
# Persist TRAIN quantile thresholds to disk (single source of truth)

from pathlib import Path

ATTR_DIR.mkdir(parents=True, exist_ok=True)
quantiles_path = ATTR_DIR / "quantiles.json"

# Reuse the train_bar_rows collected in Step 9 cell if present; otherwise rebuild quickly.
try:
    _train_rows = train_bar_rows  # noqa: F821
except NameError:
    _train_rows = []

if not _train_rows:
    # rebuild from a moderate sample to avoid huge runtime here
    rng = random.Random(0)
    train_keys = load_split(SPLITS_DIR / "train.jsonl")
    sample_keys = [train_keys[i] for i in rng.sample(range(len(train_keys)), 200)]

    for zip_name, member in sample_keys:
        b = load_midi_from_zip(DATA_DIR / zip_name, member)
        b2 = strip_leading_silence(b)
        ids_local = tokenize_midi_bytes(b2)
        chunks_local = split_into_24_bar_chunks(ids_local, BAR_ID, 24)
        for ch in chunks_local:
            xb = enforce_block_size(ch, PAD_ID, 1024)
            if xb is None:
                continue
            _train_rows.extend(
                compute_raw_attributes_per_bar_from_tokens(xb, bar_token_id=BAR_ID, bars_per_sample=24)
            )

thresholds_disk = fit_quantile_thresholds(_train_rows, num_bins=8)

payload = {
    "num_bins": 8,
    "attribute_names": ATTRIBUTE_NAMES,
    "thresholds": thresholds_disk,
}

quantiles_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("wrote:", quantiles_path)
print("polyphony thresholds:", thresholds_disk["polyphony_rate"])

# From here on, use `thresholds_disk` loaded from this file.


wrote: /Users/kshoker/Desktop/CPSC 440/project/artifacts/attributes/quantiles.json
polyphony thresholds: [0.0625, 1.3125, 2.25, 2.9375, 3.5, 4.4375, 5.9375]


## Parallel full preprocessing (all CPU cores + progress bars)

This section replaces the slow single-thread loop with a **single-writer / multi-process** pipeline:

- Workers (ProcessPool): compute examples for one `(zip, member)`
- Main process: writes `.txt`, writes temporary `.pt` shards, then merges into one `<split>.pt`
- Progress: `tqdm` per split

This runs with **restart / no-partial-overwrite** semantics:

- If `train.pt`/`train.txt` (etc.) already exist, they will be **deleted** and regenerated.
- If a previous run left a `_tmp_<split>/` folder behind, it will be **deleted** and regenerated.

So you’ll never accidentally mix partial output with new output.


In [85]:
import os
import math
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

from tqdm.auto import tqdm


def _worker_init(
    pitch_range: tuple[int, int],
    num_velocities: int,
    bar_id: int,
    pad_id: int,
    quantiles_json: str,
) -> None:
    """Initializer for worker processes (avoids re-creating tokenizer per task)."""
    global W_TOKENIZER, W_THRESHOLDS, W_ID_TO_TOKEN, W_BAR_ID, W_PAD_ID, W_PITCH_RANGE

    import json as _json
    import inspect as _inspect
    import miditok as _miditok
    from miditok import REMI as _REMI, TokenizerConfig as _TokenizerConfig

    W_BAR_ID = bar_id
    W_PAD_ID = pad_id
    W_PITCH_RANGE = pitch_range

    # Build TokenizerConfig matching our 163-token grammar (same as Step 4)
    sig = _inspect.signature(_TokenizerConfig)
    params = set(sig.parameters.keys())

    cfg_kwargs = dict(
        pitch_range=pitch_range,
        beat_res={(0, 4): 4, (4, 8): 2},
        num_velocities=num_velocities,
        use_time_signatures=False,
        use_tempos=False,
        use_sustain_pedals=False,
    )
    if "special_tokens" in params:
        cfg_kwargs["special_tokens"] = ["PAD", "BOS"]
    if "use_pitchdrum_tokens" in params:
        cfg_kwargs["use_pitchdrum_tokens"] = False
    if "sustain_pedal_duration" in params:
        cfg_kwargs["sustain_pedal_duration"] = True
    for k, v in {
        "use_chords": False,
        "use_rests": False,
        "use_programs": False,
        "use_pitch_bends": False,
        "use_pitch_intervals": False,
    }.items():
        if k in params:
            cfg_kwargs[k] = v

    W_TOKENIZER = _REMI(_TokenizerConfig(**cfg_kwargs))
    W_ID_TO_TOKEN = {i: s for (s, i) in W_TOKENIZER.vocab.items()}

    q = _json.loads(open(quantiles_json, "r", encoding="utf-8").read())
    W_THRESHOLDS = q["thresholds"]


def _worker_tokenize(midi_bytes: bytes) -> list[int]:
    import tempfile

    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp:
        tmp.write(midi_bytes)
        tmp.flush()
        seq = W_TOKENIZER.encode(tmp.name)

    if isinstance(seq, list):
        if len(seq) != 1:
            raise RuntimeError(f"Expected 1 token stream, got {len(seq)}")
        seq = seq[0]

    ids = getattr(seq, "ids", None)
    if ids is None:
        raise RuntimeError("miditok returned an object without `.ids`")
    return [int(x) for x in ids]


def _worker_strip_silence(midi_bytes: bytes) -> bytes:
    # reuse same logic as notebook but fully self-contained in worker
    import tempfile
    from miditoolkit import MidiFile

    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp:
        tmp.write(midi_bytes)
        tmp.flush()
        midi = MidiFile(tmp.name)

    min_start = None
    for inst in midi.instruments:
        for n in inst.notes:
            if min_start is None or n.start < min_start:
                min_start = n.start

    if min_start is None or min_start <= 0:
        return midi_bytes

    shift = min_start
    for inst in midi.instruments:
        for n in inst.notes:
            n.start = max(0, n.start - shift)
            n.end = max(0, n.end - shift)
        for cc in getattr(inst, "control_changes", []) or []:
            cc.time = max(0, cc.time - shift)
        for pb in getattr(inst, "pitch_bends", []) or []:
            pb.time = max(0, pb.time - shift)

    for tempo in getattr(midi, "tempo_changes", []) or []:
        tempo.time = max(0, tempo.time - shift)
    for ts in getattr(midi, "time_signature_changes", []) or []:
        ts.time = max(0, ts.time - shift)
    for ks in getattr(midi, "key_signature_changes", []) or []:
        ks.time = max(0, ks.time - shift)

    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp2:
        midi.dump(tmp2.name)
        tmp2.flush()
        tmp2.seek(0)
        return tmp2.read()


def _worker_transpose(midi_bytes: bytes, semitones: int) -> bytes | None:
    if semitones == 0:
        return midi_bytes

    import tempfile
    from miditoolkit import MidiFile

    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp:
        tmp.write(midi_bytes)
        tmp.flush()
        midi = MidiFile(tmp.name)

    lo, hi = W_PITCH_RANGE
    for inst in midi.instruments:
        if getattr(inst, "is_drum", False):
            continue
        for n in inst.notes:
            p = n.pitch + semitones
            if p < lo or p > hi:
                return None

    for inst in midi.instruments:
        if getattr(inst, "is_drum", False):
            continue
        for n in inst.notes:
            n.pitch += semitones

    with tempfile.NamedTemporaryFile(suffix=".mid", delete=True) as tmp2:
        midi.dump(tmp2.name)
        tmp2.flush()
        tmp2.seek(0)
        return tmp2.read()


def _worker_split_24bars(ids: list[int], bars_per_sample: int = 24) -> list[list[int]]:
    bar_starts = [i for i, t in enumerate(ids) if t == W_BAR_ID]
    if not bar_starts:
        return []

    chunks = []
    for k in range(0, len(bar_starts) - bars_per_sample + 1, bars_per_sample):
        start = bar_starts[k]
        end = bar_starts[k + bars_per_sample] if (k + bars_per_sample) < len(bar_starts) else len(ids)
        ch = ids[start:end]
        if ch and ch[0] == W_BAR_ID and sum(1 for t in ch if t == W_BAR_ID) == bars_per_sample:
            chunks.append(ch)
    return chunks


def _worker_enforce_block(ch: list[int], block_size: int = 1024) -> list[int] | None:
    if len(ch) > block_size:
        return None
    if len(ch) == block_size:
        return ch
    return ch + [W_PAD_ID] * (block_size - len(ch))


def _worker_build_xy(x: list[int]) -> tuple[list[int], list[int]]:
    y = x[1:] + [W_PAD_ID]
    return x, y


def _worker_bar_indices(x: list[int]) -> list[int]:
    bar_idx = []
    cur = -1
    for tok in x:
        if tok == W_BAR_ID:
            cur += 1
        if cur < 0:
            cur = 0
        if cur > 23:
            raise ValueError("Found >24 bars")
        bar_idx.append(cur)
    if cur != 23:
        raise ValueError(f"Expected end bar 23, got {cur}")
    return bar_idx


def _worker_bin_attrs(raw_24x4: list[list[float]]) -> list[list[int]]:
    import numpy as _np

    arr = _np.asarray(raw_24x4, dtype=float)
    out = _np.zeros_like(arr, dtype=int)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        th = _np.asarray(W_THRESHOLDS[name], dtype=float)
        out[:, j] = _np.digitize(arr[:, j], th, right=False)
    return out.astype(int).tolist()


def _worker_compute_raw_attrs_from_tokens(x_1024: list[int]) -> list[list[float]]:
    import numpy as _np

    onsets = [0 for _ in range(24)]
    pos_has_onset = [set() for _ in range(24)]
    velocities = [[] for _ in range(24)]
    global_intervals: list[tuple[float, float]] = []

    cur_bar = -1
    cur_pos = 0

    def parse_pos(tok: str) -> int:
        return int(tok.split("_", 1)[1])

    def parse_vel(tok: str) -> int:
        return int(tok.split("_", 1)[1])

    def parse_dur(tok: str) -> float:
        raw = tok.split("_", 1)[1]
        try:
            return float(raw)
        except Exception:
            parts = raw.split(".")
            if len(parts) == 3:
                a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
                return float(a + (b / c))
            raise

    i = 0
    while i < len(x_1024):
        tid = x_1024[i]
        tok = W_ID_TO_TOKEN.get(tid)
        if tok is None:
            i += 1
            continue
        typ = tok.split("_", 1)[0]

        if tid == W_BAR_ID or typ == "Bar":
            cur_bar += 1
            cur_pos = 0
            i += 1
            continue

        if cur_bar < 0:
            i += 1
            continue
        if cur_bar >= 24:
            break

        if typ == "Position":
            cur_pos = parse_pos(tok)
            i += 1
            continue

        if typ == "Pitch" and i + 2 < len(x_1024):
            vtok = W_ID_TO_TOKEN.get(x_1024[i + 1], "")
            dtok = W_ID_TO_TOKEN.get(x_1024[i + 2], "")
            if vtok.startswith("Velocity_") and dtok.startswith("Duration_"):
                vel = parse_vel(vtok)
                dur = parse_dur(dtok)

                onsets[cur_bar] += 1
                pos_has_onset[cur_bar].add(cur_pos)
                velocities[cur_bar].append(vel)

                s = cur_bar * 4.0 + (cur_pos * 0.25)
                e = s + dur
                global_intervals.append((s, e))

                i += 3
                continue

        i += 1

    attrs = []
    for b in range(24):
        note_density = float(onsets[b])
        rhythmic_intensity = float(len(pos_has_onset[b]) / 16.0)
        vel_std = float(_np.std(_np.asarray(velocities[b], dtype=float), ddof=0)) if velocities[b] else 0.0

        poly_samples = []
        for p in range(16):
            t = b * 4.0 + (p * 0.25)
            active = 0
            for s, e in global_intervals:
                if s <= t < e:
                    active += 1
            poly_samples.append(active)
        polyphony_rate = float(sum(poly_samples) / 16.0)

        attrs.append([polyphony_rate, rhythmic_intensity, vel_std, note_density])

    return attrs


def process_piece(zip_path: str, member: str) -> dict:
    """Worker task: preprocess one piece into examples."""
    import zipfile as _zipfile

    with _zipfile.ZipFile(zip_path, "r") as z:
        b = z.read(member)

    b = _worker_strip_silence(b)

    # for hours estimate: compute bar count from base tokens
    ids_base = _worker_tokenize(b)
    bar_count = sum(1 for t in ids_base if t == W_BAR_ID)

    examples = []
    token_lines = []

    chunks_total = 0
    chunks_kept = 0
    transpositions_kept = 0

    for st in range(12):
        bt = _worker_transpose(b, st)
        if bt is None:
            continue
        transpositions_kept += 1

        ids = _worker_tokenize(bt)
        chunks = _worker_split_24bars(ids, 24)
        for ch in chunks:
            chunks_total += 1
            xb = _worker_enforce_block(ch, 1024)
            if xb is None:
                continue
            chunks_kept += 1

            X, Y = _worker_build_xy(xb)
            BI = _worker_bar_indices(X)
            raw = _worker_compute_raw_attrs_from_tokens(X)
            attrs = _worker_bin_attrs(raw)

            examples.append({"X": X, "Y": Y, "bar_indices": BI, "attributes": attrs})
            token_lines.append(" ".join(W_ID_TO_TOKEN.get(int(i), "UNK") for i in X))

    return {
        "member": member,
        "bar_count": bar_count,
        "examples": examples,
        "token_lines": token_lines,
        "chunks_total": chunks_total,
        "chunks_kept": chunks_kept,
        "transpositions_kept": transpositions_kept,
    }


### Notes

- If you want a quick test run first, set `MAX_PIECES = 10` in the cell above and run again.
- If you stop mid-run, just rerun the same cell; it will restart cleanly.


In [ ]:
import shutil
import torch


def _rm_if_exists(p: Path) -> None:
    if p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)


def run_split_parallel(
    split_name: str,
    *,
    thresholds_json: Path,
    out_pt: Path,
    out_txt: Path,
    tmp_dir: Path,
    shard_size: int = 4096,
    max_pieces: int | None = None,
) -> dict:
    # overwrite semantics
    _rm_if_exists(out_pt)
    _rm_if_exists(out_txt)
    _rm_if_exists(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    out_pt.parent.mkdir(parents=True, exist_ok=True)

    keys = load_split(SPLITS_DIR / f"{split_name}.jsonl")
    if max_pieces is not None:
        keys = keys[:max_pieces]

    # We’ll write temporary shard files to avoid holding everything in memory.
    shard_idx = 0
    buf_X: list[list[int]] = []
    buf_Y: list[list[int]] = []
    buf_BI: list[list[int]] = []
    buf_A: list[list[list[int]]] = []

    pieces_done = 0
    chunks_total = 0
    chunks_kept = 0
    total_tokens = 0
    total_seconds = 0.0

    # deterministic: keep submission order and write results in the same order
    futures = {}

    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(
        max_workers=os.cpu_count() or 4,
        mp_context=ctx,
        initializer=_worker_init,
        initargs=(
            PITCH_RANGE,
            NUM_VELOCITIES,
            BAR_ID,
            PAD_ID,
            str(thresholds_json),
        ),
    ) as ex:
        for idx, (zip_name, member) in enumerate(keys):
            zip_path = str(DATA_DIR / zip_name)
            futures[ex.submit(process_piece, zip_path, member)] = idx

        # buffer results by idx for deterministic writes
        pending: dict[int, dict] = {}
        next_idx = 0

        with out_txt.open("w", encoding="utf-8") as ftxt:
            pbar = tqdm(total=len(keys), desc=f"{split_name} pieces")

            for fut in as_completed(futures):
                idx = futures[fut]
                res = fut.result()
                pending[idx] = res

                # write any completed results in-order
                while next_idx in pending:
                    r = pending.pop(next_idx)
                    next_idx += 1

                    pieces_done += 1
                    chunks_total += r["chunks_total"]
                    chunks_kept += r["chunks_kept"]

                    # hours estimate: use seconds-per-bar from base piece duration
                    # compute duration in main process (tempo-aware) once
                    # (read bytes here only for duration; avoids sending big data back)
                    zip_name = keys[pieces_done - 1][0]
                    member = r["member"]
                    b = load_midi_from_zip(DATA_DIR / zip_name, member)
                    b = strip_leading_silence(b)
                    dur_sec_piece = midi_duration_seconds(b)
                    bar_count = r["bar_count"] or 0
                    if bar_count > 0:
                        sec_per_bar = dur_sec_piece / bar_count
                    else:
                        sec_per_bar = 0.0

                    # append examples + write txt
                    for exm, line in zip(r["examples"], r["token_lines"]):
                        buf_X.append(exm["X"])
                        buf_Y.append(exm["Y"])
                        buf_BI.append(exm["bar_indices"])
                        buf_A.append(exm["attributes"])
                        ftxt.write(line + "\n")

                        total_tokens += 1024
                        total_seconds += 24 * sec_per_bar

                        if len(buf_X) >= shard_size:
                            shard_path = tmp_dir / f"shard_{shard_idx:05d}.pt"
                            torch.save(
                                {
                                    "X": torch.tensor(buf_X, dtype=torch.long),
                                    "Y": torch.tensor(buf_Y, dtype=torch.long),
                                    "bar_indices": torch.tensor(buf_BI, dtype=torch.long),
                                    "attributes": torch.tensor(buf_A, dtype=torch.long),
                                },
                                shard_path,
                            )
                            shard_idx += 1
                            buf_X, buf_Y, buf_BI, buf_A = [], [], [], []

                    pbar.update(1)
                    pbar.set_postfix(
                        examples=total_tokens // 1024,
                        kept_ratio=(chunks_kept / chunks_total) if chunks_total else 0.0,
                    )

            pbar.close()

    # flush remainder
    if buf_X:
        shard_path = tmp_dir / f"shard_{shard_idx:05d}.pt"
        torch.save(
            {
                "X": torch.tensor(buf_X, dtype=torch.long),
                "Y": torch.tensor(buf_Y, dtype=torch.long),
                "bar_indices": torch.tensor(buf_BI, dtype=torch.long),
                "attributes": torch.tensor(buf_A, dtype=torch.long),
            },
            shard_path,
        )
        shard_idx += 1

    # merge shards into single pt
    shard_files = sorted(tmp_dir.glob("shard_*.pt"))
    Xs = []
    Ys = []
    BIs = []
    As = []
    for sp in tqdm(shard_files, desc=f"merge {split_name}"):
        sh = torch.load(sp, map_location="cpu")
        Xs.append(sh["X"])
        Ys.append(sh["Y"])
        BIs.append(sh["bar_indices"])
        As.append(sh["attributes"])

    X = torch.cat(Xs, dim=0) if Xs else torch.empty((0, 1024), dtype=torch.long)
    Y = torch.cat(Ys, dim=0) if Ys else torch.empty((0, 1024), dtype=torch.long)
    BI = torch.cat(BIs, dim=0) if BIs else torch.empty((0, 1024), dtype=torch.long)
    A = torch.cat(As, dim=0) if As else torch.empty((0, 24, 4), dtype=torch.long)

    meta = {
        "split": split_name,
        "num_examples": int(X.shape[0]),
        "tokens_per_example": 1024,
        "total_tokens": int(X.shape[0]) * 1024,
        "estimated_seconds": float(total_seconds),
        "estimated_hours": float(total_seconds / 3600.0),
        "pieces_seen": int(len(keys)),
        "transpositions_attempted": 12,
        "chunks_total": int(chunks_total),
        "chunks_kept": int(chunks_kept),
        "kept_ratio": float(chunks_kept / chunks_total) if chunks_total else 0.0,
    }

    torch.save({"X": X, "Y": Y, "bar_indices": BI, "attributes": A, "meta": meta}, out_pt)
    print("wrote:", out_pt)
    print("wrote:", out_txt)

    return meta


# --- run full preprocessing (overwrite) ---

THRESH_JSON = ATTR_DIR / "quantiles.json"

MAX_PIECES = None  # set to an int for a smaller test run

meta_train = run_split_parallel(
    "train",
    thresholds_json=THRESH_JSON,
    out_pt=PREPROC_DIR / "train.pt",
    out_txt=PREPROC_DIR / "train.txt",
    tmp_dir=PREPROC_DIR / "_tmp_train",
    shard_size=4096,
    max_pieces=MAX_PIECES,
)
meta_val = run_split_parallel(
    "val",
    thresholds_json=THRESH_JSON,
    out_pt=PREPROC_DIR / "val.pt",
    out_txt=PREPROC_DIR / "val.txt",
    tmp_dir=PREPROC_DIR / "_tmp_val",
    shard_size=4096,
    max_pieces=MAX_PIECES,
)
meta_test = run_split_parallel(
    "test",
    thresholds_json=THRESH_JSON,
    out_pt=PREPROC_DIR / "test.pt",
    out_txt=PREPROC_DIR / "test.txt",
    tmp_dir=PREPROC_DIR / "_tmp_test",
    shard_size=4096,
    max_pieces=MAX_PIECES,
)

print("train:", meta_train)
print("val:", meta_val)
print("test:", meta_test)


train pieces:   0%|          | 0/6809 [00:00<?, ?it/s]

In [ ]:
# Report dataset sizes (hours + tokens) from saved .pt metadata

import torch


def load_meta(pt_path: Path) -> dict:
    obj = torch.load(pt_path, map_location="cpu")
    return obj.get("meta", {})


meta_train2 = load_meta(PREPROC_DIR / "train.pt")
meta_val2 = load_meta(PREPROC_DIR / "val.pt")
meta_test2 = load_meta(PREPROC_DIR / "test.pt")

for m in [meta_train2, meta_val2, meta_test2]:
    split = m.get("split")
    ex = m.get("num_examples")
    toks = m.get("total_tokens")
    hrs = m.get("estimated_hours")
    kept = m.get("chunks_kept")
    tot = m.get("chunks_total")
    ratio = (kept / tot) if (kept and tot) else None

    print(
        split,
        "examples=", ex,
        "total_tokens=", toks,
        "estimated_hours=", hrs,
        "kept_ratio=", ratio,
    )
